In [1]:
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
from one.api import ONE
from brainbox.population.decode import get_spike_counts_in_bins
from brainbox.io.one import SpikeSortingLoader, SessionLoader
from brainbox.ephys_plots import plot_brain_regions
from iblatlas.atlas import AllenAtlas
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units
from brainwidemap.bwm_loading import merge_probes
from brainbox.behavior.training import compute_performance, plot_psychometric, plot_reaction_time
from brainbox.task.trials import find_trial_ids
from brainbox.io.one import SessionLoader
from pathlib import Path
from brainbox.task.trials import get_event_aligned_raster, get_psth
from brainbox.singlecell import bin_spikes2D
import numpy as np
from iblatlas.atlas import BrainRegions
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
import itertools
import pickle as pkl
from tqdm import tqdm
from pathlib import Path
import warnings
from sklearn.ensemble import RandomForestClassifier
from ibl_info.prepare_data_pid import (
    cleaned_regions_flags,
    get_new_cinc_intervals,
    prepare_ephys_data,
)
from ibl_info.utils import (
    alternate_discretize,
    compute_mutual_information,
    compute_pid,
    compute_trivariate_mi,
    FIRING_RATE,
    discretize,
    discretize_keeping_zeros,
    equipopulated_binning,
)
import os
import concurrent.futures
import functools
import random
from ibl_info.utils import check_config

In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
important_regions = [
    "VISp",
    "MOs",
    "SSp-ul",
    "ACAd",
    "PL",
    "CP",
    "VPM",
    "MG",
    "LGd",
    "ZI",
    "SNr",
    "MRN",
    "SCm",
    "PAG",
    "APN",
    "RN",
    "PPN",
    "PRNc",
    "PRNr",
    "GRN",
    "IRN",
    "PGRN",
    "CUL4 5",
    "SIM",
    "IP",
]

In [5]:
# instead of by region, it makes more sense to run by pairs

In [6]:
one = ONE(
    base_url="https://openalyx.internationalbrainlab.org",
    password="international",
    silent=True,
    username="intbrainlab",
)
unit_df = bwm_units(one)

Loading bwm_query results from fixtures/2023_12_bwm_release.csv
d16d0b38d392b18c0ce8b615ec89d60d7c901df2eeb3432986b62130af28ef01


In [7]:
from ibl_info.decoder_pid_pairs import filter_eids_for_region_pair

In [ ]:
from ibl_info.decoder_pid_wifi import region_combinations


unit_df = bwm_units(one)
region_pairs = region_combinations(len(important_regions))
all_tasks_to_run = []
for ra, rb in region_pairs:
    eids = filter_eids_for_region_pair(unit_df, important_regions[ra], important_regions[rb])
    for eid in tqdm(eids):
        all_tasks_to_run.append((eid, important_regions[ra], important_regions[rb], "stim"))

In [ ]:
all_tasks_to_run 

[('09b2c4d1-058d-4c84-9fd4-97530f85baf6', 'VISp', 'MG', 'stim'),
 ('0cf6d255-8f2f-463e-84fb-c54bacb79f51', 'VISp', 'MG', 'stim'),
 ('d901aff5-2250-467a-b4a1-0cb9729df9e2', 'VISp', 'MG', 'stim'),
 ('07dc4b76-5b93-4a03-82a0-b3d9cc73f412', 'VISp', 'MRN', 'stim'),
 ('0c828385-6dd6-4842-a702-c5075f5f5e81', 'VISp', 'MRN', 'stim'),
 ('111c1762-7908-47e0-9f40-2f2ee55b6505', 'VISp', 'MRN', 'stim'),
 ('5ae68c54-2897-4d3a-8120-426150704385', 'VISp', 'MRN', 'stim'),
 ('5c0c560e-9e1f-45e9-b66e-e4ee7855be84', 'VISp', 'MRN', 'stim'),
 ('5d01d14e-aced-4465-8f8e-9a1c674f62ec', 'VISp', 'MRN', 'stim'),
 ('6ab9d98c-b1e9-4574-b8fe-b9eec88097e0', 'VISp', 'MRN', 'stim'),
 ('7cc74598-9c1b-436b-84fa-0bf89f31adf6', 'VISp', 'MRN', 'stim'),
 ('85dc2ebd-8aaf-46b0-9284-a197aee8b16f', 'VISp', 'MRN', 'stim'),
 ('8a3a0197-b40a-449f-be55-c00b23253bbf', 'VISp', 'MRN', 'stim'),
 ('90c61c38-b9fd-4cc3-9795-29160d2f8e55', 'VISp', 'MRN', 'stim'),
 ('90d1e82c-c96f-496c-ad4e-ee3f02067f25', 'VISp', 'MRN', 'stim'),
 ('91bac580-7